## Cell 1 — Install dependenciesm

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q --upgrade bitsandbytes
!pip install -q \
    transformers==4.46.0 \
    peft==0.13.0 \
    trl==0.11.4 \
    accelerate==0.34.0 \
    datasets==2.21.0
!pip install -q --upgrade wandb

print("Dependencies installed")

## Cell 2 — Verify GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
print("Torch version:", torch.__version__)


## Cell 3 — All imports

In [ ]:
import os
import torch
import wandb
import logging
import numpy as np
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModel,
)
from trl import SFTTrainer
from kaggle_secrets import UserSecretsClient

torch.manual_seed(42)
np.random.seed(42)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
print("All imports successful")

## Cell 4 — Login to W&B and HuggingFace

In [ ]:
# In Kaggle: loaded via UserSecretsClient
# Locally: set HF_TOKEN and WANDB_API_KEY as environment variables
# wandb.login(key=os.environ.get("WANDB_API_KEY"))
# login(token=os.environ.get("HF_TOKEN"))

secrets = UserSecretsClient()

wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

hf_token = secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)

print("W&B and HuggingFace login successful")

## Cell 5 — Config

In [ ]:
# Replace YOUR_HF_USERNAME below with your actual HuggingFace username
HF_USERNAME = "YOUR_HF_USERNAME"

config = {
    "model": {
        "name": "microsoft/Phi-3-mini-4k-instruct",
        "max_seq_length": 1024,
    },
    "quantization": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": True,
    },
    "lora": {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": [
            "q_proj", "k_proj", "v_proj",
            "o_proj", "gate_proj", "up_proj", "down_proj"
        ],
    },
    "training": {
        "output_dir": "/kaggle/working/outputs/sft",
        "num_train_epochs": 1,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "gradient_checkpointing": True,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "weight_decay": 0.001,
        "fp16": True,
        "logging_steps": 10,
        "eval_steps": 200,
        "save_steps": 200,
        "save_total_limit": 3,
        "load_best_model_at_end": True,
        "report_to": "wandb",
        "run_name": "alignforge-sft",
        "dataloader_pin_memory": False,
        "remove_unused_columns": False,
    },
    "dataset": {
        "name": "databricks/databricks-dolly-15k",
        "train_split": 0.9,
        "max_seq_length": 1024,
    }
}

print("Config loaded")
print(f"HuggingFace username: {HF_USERNAME}")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Memory allocator configured")

## Cell 6 — VRAM utility

In [ ]:
def get_vram():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free = total - reserved
    print(f"Allocated: {allocated:.2f}GB | Reserved: {reserved:.2f}GB | Free: {free:.2f}GB | Total: {total:.2f}GB")

print("Before model load:")
get_vram()

## Cell 7 — Load tokenizer and model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

logger.info("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    config["model"]["name"],
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "right"

logger.info("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    config["model"]["name"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
model.config.use_cache = False

print("After model load:")
get_vram()
print("Model loaded successfully")

## Cell 8 — Apply LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=config["lora"]["r"],
    lora_alpha=config["lora"]["lora_alpha"],
    lora_dropout=config["lora"]["lora_dropout"],
    target_modules=config["lora"]["target_modules"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("After LoRA:")
get_vram()

## Cell 9 — Prepare dataset

In [ ]:
def format_phi3_prompt(sample):
    context = sample.get("context", "").strip()
    instruction = sample["instruction"].strip()
    response = sample["response"].strip()
    
    if context:
        text = f"<|user|>\n{instruction}\n\n{context}<|end|>\n<|assistant|>\n{response}<|end|>"
    else:
        text = f"<|user|>\n{instruction}<|end|>\n<|assistant|>\n{response}<|end|>"
    
    return {"text": text}

logger.info("Loading dataset...")
raw_dataset = load_dataset(config["dataset"]["name"])

logger.info("Formatting prompts...")
formatted = raw_dataset["train"].map(format_phi3_prompt)

def tokenize_for_filter(sample):
    tokens = tokenizer(sample["text"], truncation=False)
    return {"token_length": len(tokens["input_ids"])}

tokenized = formatted.map(tokenize_for_filter)

before = len(tokenized)
tokenized = tokenized.filter(lambda x: x["token_length"] >= 10)
tokenized = tokenized.filter(lambda x: x["token_length"] < config["model"]["max_seq_length"])
after = len(tokenized)
logger.info(f"Filtered {before - after} samples. Remaining: {after}")

split = tokenized.train_test_split(test_size=1 - config["dataset"]["train_split"], seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

## Cell 10 — Data collator

In [ ]:
class SFTDataCollator:
    def __init__(self, tokenizer, max_length=2048):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.assistant_token_ids = tokenizer.encode(
            "<|assistant|>", add_special_tokens=False
        )
    
    def _find_assistant_start(self, input_ids):
        target = self.assistant_token_ids
        for i in range(len(input_ids) - len(target) + 1):
            if input_ids[i:i+len(target)] == target:
                return i + len(target)
        return None
    
    def __call__(self, batch):
        texts = [item["text"] for item in batch]
        tokenized = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors="pt",
        )
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        labels = input_ids.clone()
        
        for i in range(len(batch)):
            ids_list = input_ids[i].tolist()
            assistant_start = self._find_assistant_start(ids_list)
            if assistant_start is not None:
                labels[i, :assistant_start] = -100
            else:
                labels[i, :] = -100
            labels[i][attention_mask[i] == 0] = -100
        
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

collator = SFTDataCollator(tokenizer, max_length=config["model"]["max_seq_length"])
print("Collator ready")

## Cell 11 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=config["training"]["output_dir"],
    num_train_epochs=config["training"]["num_train_epochs"],
    per_device_train_batch_size=config["training"]["per_device_train_batch_size"],
    per_device_eval_batch_size=config["training"]["per_device_eval_batch_size"],
    gradient_accumulation_steps=config["training"]["gradient_accumulation_steps"],
    gradient_checkpointing=config["training"]["gradient_checkpointing"],
    learning_rate=config["training"]["learning_rate"],
    lr_scheduler_type=config["training"]["lr_scheduler_type"],
    warmup_ratio=config["training"]["warmup_ratio"],
    weight_decay=config["training"]["weight_decay"],
    fp16=config["training"]["fp16"],
    logging_steps=config["training"]["logging_steps"],
    eval_strategy="steps",
    eval_steps=config["training"]["eval_steps"],
    save_strategy="steps",
    save_steps=config["training"]["save_steps"],
    save_total_limit=config["training"]["save_total_limit"],
    load_best_model_at_end=config["training"]["load_best_model_at_end"],
    report_to=config["training"]["report_to"],
    run_name=config["training"]["run_name"],
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    max_seq_length=config["model"]["max_seq_length"],
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

print("Trainer built")
print("Starting training...")
train_result = trainer.train()
print(f"Training complete. Loss: {train_result.training_loss:.4f}")

## Cell 12 — Save and merge

In [ ]:
adapter_path = "/kaggle/working/outputs/sft/adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

logger.info("Loading base model for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    config["model"]["name"],
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

merged_path = "/kaggle/working/outputs/sft/merged"
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print(f"Merged model saved to {merged_path}")

## Cell 13 — Push to HuggingFace

In [ ]:
merged_model.push_to_hub(f"{HF_USERNAME}/alignforge-phi3-sft")
tokenizer.push_to_hub(f"{HF_USERNAME}/alignforge-phi3-sft")
print(f"Pushed to: huggingface.co/{HF_USERNAME}/alignforge-phi3-sft")